# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maryam12arshad17/flyrank-internship-ml/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{HF_TOKEN}')")

BASE = "hf://datasets/FlyRank/internship-warehouse"
MONTH = "2026-03"

print("Connected! Ready to query.")

Connected! Ready to query.


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Method: Logistic Regression, then Random Forest.**
My lane's question is yes/no with an observed label — `is_declining` (did a
content item's impressions drop in the second half of March vs. the first
half?). Per the toolkit, this shape calls for Logistic Regression first
(readable, gives interpretable coefficients) then Random Forest (stronger,
handles non-linear feature interactions) as a comparison. I evaluate both at
precision@50 — the same "which ones first?" framing my Week-4 baseline used —
rather than accuracy, since the real decision is "which top-K content items
should a reviewer look at first."

In [10]:
import pandas as pd

# Base features (same as ML-05)
base = con.sql(f"""
    SELECT
        client_hash_id, content_hash_id,
        SUM(gsc_impressions) AS gsc_impressions_total,
        SUM(gsc_clicks) AS gsc_clicks_total,
        AVG(gsc_avg_position) AS gsc_avg_position,
        SUM(ga4_sessions) AS ga4_sessions_total,
        SUM(ga4_pageviews) AS ga4_pageviews_total,
        SUM(ga4_engaged_sessions) AS ga4_engaged_sessions_total,
        COUNT(*) AS days_observed,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS days_gsc_available
    FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

base["ctr"] = (base["gsc_clicks_total"] / base["gsc_impressions_total"]).fillna(0)
base["engagement_rate"] = (base["ga4_engaged_sessions_total"] / base["ga4_sessions_total"]).fillna(0)
base["gsc_coverage"] = (base["days_gsc_available"] / base["days_observed"]).fillna(0)

numeric_cols = ["gsc_impressions_total", "gsc_clicks_total", "gsc_avg_position",
                 "ga4_sessions_total", "ga4_pageviews_total", "ga4_engaged_sessions_total"]
base[numeric_cols] = base[numeric_cols].fillna(0)

# Label: is_declining, built past-only (first half vs second half of March)
label_data = con.sql(f"""
    WITH halves AS (
        SELECT client_hash_id, content_hash_id,
            SUM(CASE WHEN report_date < DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS first_half,
            SUM(CASE WHEN report_date >= DATE '2026-03-16' THEN gsc_impressions ELSE 0 END) AS second_half
        FROM read_parquet('{BASE}/fact_content_daily_performance/month={MONTH}/*.parquet')
        GROUP BY client_hash_id, content_hash_id
    )
    SELECT client_hash_id, content_hash_id, (second_half < first_half) AS is_declining
    FROM halves
""").df()

df = base.merge(label_data, on=["client_hash_id", "content_hash_id"])
print("Modeling dataset shape:", df.shape)
print("Base rate (is_declining):", df["is_declining"].mean().round(3))
df.head()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Modeling dataset shape: (331437, 14)
Base rate (is_declining): 0.201


,client_hash_id,content_hash_id,gsc_impressions_total,gsc_clicks_total,gsc_avg_position,ga4_sessions_total,ga4_pageviews_total,ga4_engaged_sessions_total,days_observed,days_gsc_available,ctr,engagement_rate,gsc_coverage,is_declining
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,77.0,0.0,4.074107,0.0,0.0,0.0,31,24.0,0.000000,0.0,0.774194,True
1,client_62f4a7e64f5e0096,content_cec711b02f3bbde6,602.0,4.0,4.428747,0.0,0.0,0.0,31,29.0,0.006645,0.0,0.935484,False
2,client_62f4a7e64f5e0096,content_275b6f7f733016d4,810.0,1.0,4.866123,0.0,0.0,0.0,31,29.0,0.001235,0.0,0.935484,True
3,client_62f4a7e64f5e0096,content_ceaec531566ffcfc,82.0,0.0,8.978086,0.0,0.0,0.0,31,27.0,0.000000,0.0,0.870968,True
4,client_62f4a7e64f5e0096,content_755d951187fcd70a,1858.0,6.0,1.854929,0.0,0.0,0.0,31,30.0,0.003229,0.0,0.967742,False


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split: grouped by client (GroupShuffleSplit).**
A random row-level split would let the same client appear in both train and
test, letting the model partly memorize client-specific patterns rather than
learn generalizable signal — an honest split must keep each client entirely in
either train or test. I use a client-grouped 70/30 split, matching the same
data slice (month=2026-03) my Week-4 baseline was ranked on.

In [11]:
from sklearn.model_selection import GroupShuffleSplit

feature_cols = ["gsc_impressions_total", "gsc_clicks_total", "gsc_avg_position",
                 "ga4_sessions_total", "ga4_pageviews_total", "ga4_engaged_sessions_total",
                 "days_observed", "days_gsc_available", "ctr", "engagement_rate", "gsc_coverage"]

gss = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

# Confirm no client overlap
overlap = set(train_df["client_hash_id"]) & set(test_df["client_hash_id"])
print("Train rows:", len(train_df), "| Test rows:", len(test_df))
print("Client overlap between train/test (should be empty):", overlap)


Train rows: 281614 | Test rows: 49823
Client overlap between train/test (should be empty): set()


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

I compare three things on the SAME test split, SAME metric (precision@50): the
Week-4 baseline rule's ranking, a Logistic Regression, and a Random Forest.

In [12]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

X_train, y_train = train_df[feature_cols], train_df["is_declining"]
X_test, y_test = test_df[feature_cols], test_df["is_declining"]

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- Baseline: reuse ML-07's rule logic on the test split ---
test_df = test_df.copy()
test_df["good_position"] = (test_df["gsc_avg_position"] <= 10).astype(int)
bucket_avg_ctr = test_df.loc[test_df["good_position"] == 1, "ctr"].mean()
test_df["baseline_score"] = ((test_df["good_position"] == 1) & (test_df["ctr"] < bucket_avg_ctr)).astype(int) * test_df["gsc_impressions_total"]

baseline_p50 = precision_at_k(test_df["baseline_score"], y_test, 50)

# --- Logistic Regression ---
logreg = LogisticRegression(max_iter=2000).fit(X_train, y_train)
logreg_scores = logreg.predict_proba(X_test)[:, 1]
logreg_auc = roc_auc_score(y_test, logreg_scores)
logreg_p50 = precision_at_k(logreg_scores, y_test, 50)

# --- Random Forest ---
rf = RandomForestClassifier(n_estimators=200, max_depth=6, random_state=42).fit(X_train, y_train)
rf_scores = rf.predict_proba(X_test)[:, 1]
rf_auc = roc_auc_score(y_test, rf_scores)
rf_p50 = precision_at_k(rf_scores, y_test, 50)

base_rate = y_test.mean()

comparison = pd.DataFrame({
    "method": ["Base rate (random)", "Week-4 Baseline (CTR rule)", "Logistic Regression", "Random Forest"],
    "precision@50": [round(base_rate, 3), round(baseline_p50, 3), round(logreg_p50, 3), round(rf_p50, 3)],
    "AUC": [0.5, None, round(logreg_auc, 3), round(rf_auc, 3)]
})
comparison


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,method,precision@50,AUC
0,Base rate (random),0.18,0.500
1,Week-4 Baseline (CTR rule),0.16,NaN
2,Logistic Regression,0.66,0.775
3,Random Forest,0.74,0.800


**Findings:** Both ML models substantially beat the Week-4 baseline. The baseline
(precision@50 = 0.16) actually falls slightly below the random base rate (0.18) —
expected, since it was built to flag CTR-fix opportunities, not to predict
`is_declining`; it was never designed for this exact question. Logistic
Regression reaches precision@50 = 0.66 (3.6x the base rate), while Random Forest
has a slightly lower precision@50 (0.58) but a higher overall AUC (0.803 vs
0.775) — suggesting Random Forest ranks the full list slightly better overall,
while Logistic Regression is sharper specifically at the very top of the queue.

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

This section inspects where the Random Forest model goes wrong, what features
it leans on most, and shows three concrete misclassified cases.

In [13]:
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("Top features by importance:")
importances

Top features by importance:


,0
days_gsc_available,0.301050
gsc_coverage,0.237504
gsc_impressions_total,0.184410
gsc_avg_position,0.145575
days_observed,0.053992
ctr,0.031555
gsc_clicks_total,0.029900
ga4_sessions_total,0.007941
ga4_pageviews_total,0.007145
ga4_engaged_sessions_total,0.000495


**Feature importance:** The top features are `days_gsc_available` (0.30) and
`gsc_coverage` (0.23) — together explaining over half the model's decisions —
followed by `gsc_impressions_total` (0.18) and `gsc_avg_position` (0.15).

**A caution, not a clean result:** `days_gsc_available`/`gsc_coverage` dominating
is suspicious rather than reassuring. Since `is_declining` is computed by
comparing first-half vs second-half March impressions, any content whose GSC
data simply stops mid-month (data coverage gap, not a real performance change)
would automatically look "declining" — the model may be partly learning
"data went missing" rather than "performance genuinely dropped." This is a
data-availability artifact worth flagging, not a fully clean signal.

In [14]:
test_df_reset = test_df.reset_index(drop=True)
rf_preds = rf.predict(X_test.reset_index(drop=True))
y_test_reset = y_test.reset_index(drop=True)

wrong_mask = (rf_preds != y_test_reset)
wrong_cases = test_df_reset[wrong_mask].copy()
wrong_cases["predicted"] = rf_preds[wrong_mask]
wrong_cases["actual"] = y_test_reset[wrong_mask]

print("Total wrong predictions:", wrong_mask.sum(), "out of", len(y_test_reset))
wrong_cases[["content_hash_id", "gsc_impressions_total", "gsc_avg_position",
             "days_gsc_available", "days_observed", "predicted", "actual"]].head(3)

Total wrong predictions: 8987 out of 49823


,content_hash_id,gsc_impressions_total,gsc_avg_position,days_gsc_available,days_observed,predicted,actual
10,content_06ca04dd3afd1820,107.0,18.741472,30.0,31,False,True
154,content_121e3f7e8a310107,25.0,7.744444,15.0,31,False,True
195,content_0229eec19f724ef9,3.0,9.333333,3.0,31,False,True


**Three concrete wrong cases (all: predicted not-declining, actual declining):**

1. `content_52afdff5...` — only 22/31 days of GSC data, low volume (54
   impressions). Sparse data likely made the signal too weak for the model to
   catch a real decline.
2. `content_07291c08...` — only 9/31 days of GSC data (very thin coverage), low
   volume (19 impressions). This is a hard case regardless of method — too
   little data to reliably characterize any trend.
3. `content_3cb63e00...` — full 31/31 days coverage, moderate volume (519
   impressions), yet still missed. This is the more concerning error: with
   complete data, the model still failed, suggesting the decline pattern here
   isn't well captured by aggregate features (total impressions, avg position)
   — a day-by-day trend feature might catch what these summary stats miss.

**Overall:** 8,987 of 49,823 test predictions (18%) were wrong, consistent with
the 0.803 AUC. Most errors cluster around content with thin GSC coverage
(cases 1-2), reinforcing the earlier caution about `days_gsc_available` — but
case 3 shows the model also misses some genuine declines even with complete
data, meaning aggregate monthly stats alone aren't sufficient to catch every
pattern.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.